# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

- Load your committed dataset.
- Apply the feature engineering, preprocessing, and train/val/test split
  consistent with the business framing you wrote in `REPORT.md`.
- Train the same model — linear regression for a regression target,
  logistic regression for a binary one — three ways, on the same split:
  scikit-learn, a from-scratch PyTorch loop (Session 5's manual approach:
  raw tensors, autograd, a manual gradient-step update), and the standard
  `torch.nn.Module` + `torch.optim` workflow.
- Evaluate all three against each other and against a naive baseline.
- Save whatever `app.py`/`app.ipynb` needs to build its dashboard without
  retraining anything.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

In [ ]:
import json
import os
import urllib.request

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
DATA_PATH = "data/telco_churn.csv"
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
ART = "artifacts"

##business costs
COST_CALL = 20    # falso positivo: llamamos a alguien que no se iba a ir
COST_CHURN = 500  # falso negativo: no llamamos y el cliente se fue

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(ART, exist_ok=True)

**Clean and upload data:**

TotalCharges comes as text and new clients have it empty so it will convert to a number and become 0 (still havent paid anything) 

In [ ]:
# Si el CSV no está en el repo, se descarga una vez (lo ideal es commitearlo)
if not os.path.exists(DATA_PATH):
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
df["Churn"] = (df["Churn"] == "Yes").astype(int)

print(df.shape)
print("Churn rate:", round(df["Churn"].mean(), 3))
df.head()

**Feature engineering:**

The decisions on what features to create and why to use them. 

In [ ]:
def engineer_features(df):
    df = df.copy()

    # 1.Number of services contracted 
    #    More services = customer is more "locked in" to the company = harder to leave
    service_cols = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
    df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)

    # 2. Tenure groups
    #    Churn risk does not decrease linearly: it is very high during the first year
    df["tenure_group"] = pd.cut(
        df["tenure"], bins=[-1, 12, 24, 48, 72],
        labels=["0-12", "13-24", "25-48", "49-72"],
    ).astype(str)

    # 5. No protection: has internet but neither online security nor tech support
    #    If something goes wrong and there is no support, the customer gets frustrated
    df["no_protection"] = (
        (df["InternetService"] != "No")
        & (df["OnlineSecurity"] == "No")
        & (df["TechSupport"] == "No")
    ).astype(int)

    # 6. New month-to-month customer: nothing is tying them down
    df["new_month_to_month"] = (
        (df["tenure"] <= 12) & (df["Contract"] == "Month-to-month")
    ).astype(int)

    return df

df = engineer_features(df)

# churn rate for each new feature
for col in ["num_services", "tenure_group", "no_protection", "new_month_to_month"]:
    print(df.groupby(col)["Churn"].mean().round(3))
    print()

**Stratified split + preprocessing:**

An 80/20 stratified split is used to keep the ~27% churn rate consistent in both train and test sets
One-hot encoding and scaling are fitted only on the training set to avoid data leakage from the test set

In [ ]:
y = df["Churn"]
X = df.drop(columns=["Churn", "customerID"])
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

num_cols = X_train_raw.select_dtypes(include="number").columns.tolist()

X_train = pd.get_dummies(X_train_raw, drop_first=True)
X_test = pd.get_dummies(X_test_raw, drop_first=True).reindex(columns=X_train.columns, fill_value=0)

scaler = StandardScaler().fit(X_train[num_cols])
X_train[num_cols] = scaler.transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

Xtr, Xte = X_train.values, X_test.values
ytr, yte = y_train.values, y_test.values
print("Train:", Xtr.shape, "Test:", Xte.shape)
print("Churn rate train/test:", round(ytr.mean(), 3), round(yte.mean(), 3))

**4a. Scikit-learn**

In [ ]:
sk = LogisticRegression(max_iter=2000)
sk.fit(Xtr, ytr)
p_sk = sk.predict_proba(Xte)[:, 1]

**4b. PyTorch manual**

In [ ]:
def train_manual(X, y, lr=0.1, epochs=2000):
    Xt = torch.tensor(X)
    yt = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    w = torch.zeros((X.shape[1], 1), requires_grad=True)
    b = torch.zeros(1, requires_grad=True)

    for _ in range(epochs):
        p = torch.sigmoid(Xt @ w + b)
        loss = -(yt * torch.log(p + 1e-7) + (1 - yt) * torch.log(1 - p + 1e-7)).mean()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_()
            b.grad.zero_()
    print("Manual final loss:", round(loss.item(), 4))
    return {"w": w.detach(), "b": b.detach()}

manual = train_manual(Xtr, ytr)
with torch.no_grad():
    p_manual = torch.sigmoid(torch.tensor(Xte) @ manual["w"] + manual["b"]).numpy().ravel()

**4c. PyTorch estándar: `nn.Module` + optimizer**

In [ ]:
class LogReg(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)  # logits; la sigmoide va dentro de la loss

tnn = LogReg(Xtr.shape[1])
opt = torch.optim.SGD(tnn.parameters(), lr=0.1)
loss_fn = nn.BCEWithLogitsLoss()
Xt = torch.tensor(Xtr)
yt = torch.tensor(ytr, dtype=torch.float32).view(-1, 1)

for _ in range(2000):
    opt.zero_grad()
    loss = loss_fn(tnn(Xt), yt)
    loss.backward()
    opt.step()
print("torch.nn final loss:", round(loss.item(), 4))

with torch.no_grad():
    p_nn = torch.sigmoid(tnn(torch.tensor(Xte))).numpy().ravel()

**5.Comparison of the 3 methods + baselines**

Business baselines: never call anyone vs. call everyone.
Agreement: maximum difference between each model’s probabilities and scikit-learn’s output.

In [ ]:
def business_cost(y_true, y_pred):
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    return fp * COST_CALL + fn * COST_CHURN

def evaluate(name, y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "model": name,
        "roc_auc": round(float(roc_auc_score(y_true, proba)), 4),
        "accuracy@0.5": round(float(accuracy_score(y_true, pred)), 4),
        "cost@0.5": business_cost(y_true, pred),
    }

metrics = [
    evaluate("sklearn", yte, p_sk),
    evaluate("manual_pytorch", yte, p_manual),
    evaluate("torch_nn", yte, p_nn),
]

n = len(yte)
baselines = {
    "never_call_cost": business_cost(yte, np.zeros(n, dtype=int)),
    "call_everyone_cost": business_cost(yte, np.ones(n, dtype=int)),
    "majority_class_accuracy": round(float((yte == 0).mean()), 4),
}

agreement = {
    "max_abs_diff_manual_vs_sklearn": round(float(np.abs(p_manual - p_sk).max()), 4),
    "max_abs_diff_nn_vs_sklearn": round(float(np.abs(p_nn - p_sk).max()), 4),
}

display(pd.DataFrame(metrics))
print("Baselines:", baselines)
print("Agreement:", agreement)

**6. Save artifacts**:

The Gradio app loads these files and never retrains.

In [ ]:
joblib.dump(sk, f"{ART}/sklearn_model.pkl")
torch.save(manual, f"{ART}/manual_pytorch.pt")
torch.save(tnn.state_dict(), f"{ART}/torch_nn.pt")
joblib.dump(
    {"scaler": scaler, "num_cols": num_cols, "feature_cols": X_train.columns.tolist()},
    f"{ART}/preprocessing.pkl",
)

test_out = X_test_raw.copy()
test_out["y_true"] = yte
test_out["p_sklearn"] = p_sk
test_out["p_manual"] = p_manual
test_out["p_torch_nn"] = p_nn
test_out.to_csv(f"{ART}/test_predictions.csv", index=False)

with open(f"{ART}/metrics.json", "w") as f:
    json.dump(
        {"metrics": metrics, "baselines": baselines, "agreement": agreement,
         "costs": {"call": COST_CALL, "churn": COST_CHURN}},
        f, indent=2,
    )
print("Safed Artifacts on", ART)